# Wollaston's Campus Market — Inventory Management System

**Store context.** Wollaston's is Northeastern's campus convenience market at Stetson East, run under the Aramark dining contract. It sells grab-and-go food, drinks, snacks, dairy/frozen, personal care, and school supplies, and it takes Dining Dollars, NUCard, and credit/debit.

**What this project is.** A SQL-only project: relational schema, triggers, stored procedures, analytical views, and BI hooks. There's no Python in this notebook. It's a directory of `.sql` files plus DAX/M for Power BI and a Tableau workbook stub, and the notebook is basically a writeup of what's in the folder.

---

## Schema

```
categories ──< products >── suppliers
                  │
              inventory
                  │
           transaction_items ── transactions ── payment_methods
                  │
       ┌──────────┴──────────┐
  stock_audit_log      reorder_alerts
      (trigger)           (trigger)
```

3NF. Sale price gets captured into `transaction_items.unit_price` at write time so historical revenue doesn't shift around when a product's list price changes later. (I learned the hard way on a previous assignment what happens if you don't do this.)

---

## What's in `sql/`

Run in order against a Postgres DB called `wollastons`:

| File | Contents |
|------|----------|
| `01_ddl_schema.sql` | tables, FKs, indexes, drop-in-reverse for re-runs |
| `02_seed_data.sql` | 7 categories, 9 suppliers, 41 products (real SKUs from the store), 4 payment methods, initial inventory |
| `03_triggers_and_procedures.sql` | `fn_decrement_inventory`, `fn_update_txn_total`, plus the `receive_stock` and `process_transaction` procedures |
| `04_analytical_views.sql` | 7 views (top selling, dead stock, low stock, category margin, payment mix, hourly trend, 30-day turnover) |
| `05_sample_queries.sql` | 10 reporting queries that pull from the views above |

```bash
for f in sql/0*.sql; do psql -U postgres -d wollastons -f "$f"; done
```

---

## A few design choices worth flagging

- **Triggers do the work, not the app layer.** Every `INSERT` into `transaction_items` decrements inventory, writes a `stock_audit_log` row, and conditionally inserts a `reorder_alerts` row. The transaction-total update is also a trigger. The app side just does one insert per line item.
- **Alerts are idempotent.** The reorder trigger only inserts if there's no unresolved alert already for that product, so re-firing it as stock keeps falling doesn't spam duplicates. (First version did spam duplicates. Not great.)
- **`receive_stock` closes the loop.** When stock is replenished, it auto-resolves the matching open reorder alert, so the alerts table reflects real outstanding work and not history.
- **Payment methods are split into `Dining Dollars` / `NUCard` / `Credit/Debit` / `Cash`** because the spend-mix question (what fraction of revenue is locked to the dining plan?) seemed like the most realistic Aramark reporting ask.

---

## BI / dashboard artifacts

These are scaffolding, not finished interactive dashboards:

- `powerbi/measures.dax` — DAX measures (Total Revenue, Gross Margin %, etc.) to paste into Power BI Desktop after pointing it at the database.
- `powerbi/queries.m` — Power Query M for the dataset connections.
- `tableau/wollastons_dashboard.twb` — Tableau workbook starter.

Wiring them up to a live database is the next step. The current files give you the calculations and the connection logic, but a reviewer would still need to open the tool to render the visuals. I didn't have time to actually deploy a live Postgres instance for the dashboards to connect to, which is the one thing I'd add if I came back to this.